In [ ]:
import os
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments,BitsAndBytesConfig
from datasets import load_dataset
from trl import SFTTrainer
from peft import AutoPeftModelForCausalLM, LoraConfig, get_peft_model, prepare_model_for_kbit_training

os.environ['HUGGING_FACE_HUB_TOKEN'] = "hfkey"

In [ ]:
import os
import torch
from unsloth import FastLanguageModel
from transformers import TextStreamer

# Set your Hugging Face Hub token if required
os.environ['HUGGING_FACE_HUB_TOKEN'] = "hfkey"

# Define model parameters
max_seq_length = 2048  # Maximum sequence length (RoPE scaling auto-handled)
dtype = None           # Auto-detect data type (or set explicitly, e.g. torch.float16)
load_in_4bit = True    # Enable 4-bit quantization for reduced memory usage

# Load the model and tokenizer using unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-7B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    prompt.format(
        "Continue the fibonnaci sequence.", # instruction
        "1, 1, 2, 3, 5, 8", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
Output = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

/tmp/ipykernel_1671057/1018750441.py:3: UserWarning: WARNING: Unsloth should be imported before trl, transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Failed to patch Gemma3ForConditionalGeneration.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.19: Fast Qwen2 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 2. Max memory: 79.138 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.22s/it]


Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Continue the fibonnaci sequence.

### Input:
1, 1, 2, 3, 5, 8

### Response:
13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181, 6765, 10946, 17711, 28657, 46368, 75077, 121393, 193497, 194649, 


In [16]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
       target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],

    lora_alpha = 32,
    lora_dropout = 0.05, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,  
    loftq_config = None, 
)

In [17]:
prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

from datasets import load_dataset
dataset = load_dataset("bkai-foundation-models/vi-alpaca", split = "train", cache_dir="/home/ltnga/.cache/huggingface/datasets")
dataset = dataset.map(formatting_prompts_func, batched = True,)

Map: 100%|██████████| 50006/50006 [00:01<00:00, 42290.19 examples/s]


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=4,  # Increase parallelism for A100
    packing=False,  # Can make training 5x faster for short sequences.
    args=TrainingArguments(
        per_device_train_batch_size=16,  # Optimize for A100 memory
        gradient_accumulation_steps=2,  # Adjust accumulation to balance memory and speed
        warmup_steps=100,  # Increase for better stability on large models
        max_steps=1000,  # Extend training steps for more substantial runs
        learning_rate=1e-4,  # Adjust LR for A100 training dynamics
        fp16=not is_bfloat16_supported(),  # Use mixed precision if BF16 is not supported
        bf16=is_bfloat16_supported(),  # Use BF16 on A100 if supported
        logging_steps=10,  # Log less frequently to reduce I/O overhead
        optim="adamw_8bit",  # Efficient optimization for large-scale training
        weight_decay=0.01,
        lr_scheduler_type="cosine",  # Cosine scheduler for smoother convergence
        seed=3407,
        output_dir="outputs",
        report_to="none",  # Use this for WandB etc
        save_strategy="steps",  # Save checkpoints regularly
        save_steps=100,  # Adjust based on training steps
        save_total_limit=2,  # Limit checkpoints to save space
    ),
)

Map (num_proc=4): 100%|██████████| 50006/50006 [00:11<00:00, 4340.10 examples/s]
max_steps is given, it will override any value given in num_train_epochs


In [23]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 50,006 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 16 | Gradient Accumulation steps = 2
\        /    Total batch size = 32 | Total steps = 10
 "-____-"     Number of trainable parameters = 161,480,704


Step,Training Loss
10,0.791300


In [27]:
# Save the fine-tuned model
output_dir = "model_stage1"
model.save_pretrained(output_dir)

# Save the tokenizer
tokenizer.save_pretrained(output_dir)

('model_stage1/tokenizer_config.json',
 'model_stage1/special_tokens_map.json',
 'model_stage1/vocab.json',
 'model_stage1/merges.txt',
 'model_stage1/added_tokens.json',
 'model_stage1/tokenizer.json')

### Benchmark the model

In [7]:
import pandas as pd
import torch
from nltk.translate.bleu_score import corpus_bleu
from rouge_score import rouge_scorer
from tqdm import tqdm
from unsloth import FastLanguageModel

# --------------------------
# Load the Models
# --------------------------
# Define the directories or identifiers for the models:
base_model_dir = "Qwen/Qwen2.5-7B"  
fine_model_dir = "/home/ltnga/NguyenTrinhTest/model_stage3"

# Load the base model (Qwen2.5-7B) and its tokenizer
base_model, base_tokenizer = FastLanguageModel.from_pretrained(base_model_dir)
FastLanguageModel.for_inference(base_model)

# Load the fine-tuned model and its tokenizer
fine_model, fine_tokenizer = FastLanguageModel.from_pretrained(fine_model_dir)
FastLanguageModel.for_inference(fine_model)

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"

# --------------------------
# Define Prompt Template
# --------------------------
answer_prompt_template = """Below is a medical context and a question based on that context.
Use the context to answer the question.

Context:
{}
Question:
{}

Answer:
"""

# --------------------------
# Load the Dataset
# --------------------------
# Dataset should have columns: "question", "answer", "context"
dataset = pd.read_csv("/home/ltnga/NguyenTrinhTest/ViMedAQA_full_dataset.csv")
benchmark_df = dataset.head(1000)

# --------------------------
# Prepare Storage for Metrics
# --------------------------
# For Base Model:
base_bleu_candidates = []
base_bleu_references = []
base_rouge1_scores = []
base_rougeL_scores = []

# For Fine-Tuned Model:
fine_bleu_candidates = []
fine_bleu_references = []
fine_rouge1_scores = []
fine_rougeL_scores = []

# Initialize ROUGE scorer
rouge_scorer_inst = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

# --------------------------
# Benchmark Loop
# --------------------------
for idx, row in tqdm(benchmark_df.iterrows(), total=len(benchmark_df)):
    question = row["question"]
    context = row["context"]
    reference = row["answer"]

    # Create the prompt for this example.
    prompt = answer_prompt_template.format(context, question)

    # --- Base Model Generation ---
    base_inputs = base_tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(device)
    base_outputs = base_model.generate(**base_inputs, max_new_tokens=100)
    base_generated_text = base_tokenizer.decode(base_outputs[0], skip_special_tokens=True).strip()
    # Assume the answer is the text after "Answer:"
    base_candidate_answer = base_generated_text.split("Answer:")[-1].strip()

    # --- Fine-Tuned Model Generation ---
    fine_inputs = fine_tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(device)
    fine_outputs = fine_model.generate(**fine_inputs, max_new_tokens=100)
    fine_generated_text = fine_tokenizer.decode(fine_outputs[0], skip_special_tokens=True).strip()
    fine_candidate_answer = fine_generated_text.split("Answer:")[-1].strip()

    # --------------------------
    # Collect Outputs and Compute Scores
    # --------------------------
    # Tokenize answers for BLEU evaluation (simple whitespace split; consider a specialized Vietnamese tokenizer)
    base_candidate_tokens = base_candidate_answer.split()
    fine_candidate_tokens = fine_candidate_answer.split()
    reference_tokens = reference.split()

    base_bleu_candidates.append(base_candidate_tokens)
    base_bleu_references.append([reference_tokens])  # BLEU expects a list of reference lists

    fine_bleu_candidates.append(fine_candidate_tokens)
    fine_bleu_references.append([reference_tokens])

    # Compute ROUGE for base model
    base_rouge = rouge_scorer_inst.score(reference, base_candidate_answer)
    base_rouge1_scores.append(base_rouge['rouge1'].fmeasure)
    base_rougeL_scores.append(base_rouge['rougeL'].fmeasure)

    # Compute ROUGE for fine-tuned model
    fine_rouge = rouge_scorer_inst.score(reference, fine_candidate_answer)
    fine_rouge1_scores.append(fine_rouge['rouge1'].fmeasure)
    fine_rougeL_scores.append(fine_rouge['rougeL'].fmeasure)

# --------------------------
# Compute Corpus-Level Metrics
# --------------------------
base_corpus_bleu = corpus_bleu(base_bleu_references, base_bleu_candidates)
fine_corpus_bleu = corpus_bleu(fine_bleu_references, fine_bleu_candidates)

base_avg_rouge1 = sum(base_rouge1_scores) / len(base_rouge1_scores)
base_avg_rougeL = sum(base_rougeL_scores) / len(base_rougeL_scores)

fine_avg_rouge1 = sum(fine_rouge1_scores) / len(fine_rouge1_scores)
fine_avg_rougeL = sum(fine_rougeL_scores) / len(fine_rougeL_scores)

# --------------------------
# Print the Benchmark Results
# --------------------------
print("Base Model (Qwen2.5-7B) Benchmark Results:")
print(f"Corpus BLEU score: {base_corpus_bleu:.4f}")
print(f"Average ROUGE-1 F1 score: {base_avg_rouge1:.4f}")
print(f"Average ROUGE-L F1 score: {base_avg_rougeL:.4f}")

print("\nFine-Tuned Model Benchmark Results:")
print(f"Corpus BLEU score: {fine_corpus_bleu:.4f}")
print(f"Average ROUGE-1 F1 score: {fine_avg_rouge1:.4f}")
print(f"Average ROUGE-L F1 score: {fine_avg_rougeL:.4f}")


==((====))==  Unsloth 2025.2.4: Fast Qwen2 patching. Transformers: 4.46.1.
   \\   /|    GPU: NVIDIA A100-SXM4-80GB. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.22s/it]


==((====))==  Unsloth 2025.2.4: Fast Qwen2 patching. Transformers: 4.46.1.
   \\   /|    GPU: NVIDIA A100-SXM4-80GB. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


100%|██████████| 1000/1000 [44:29<00:00,  2.67s/it] 


Base Model (Qwen2.5-7B) Benchmark Results:
Corpus BLEU score: 0.2564
Average ROUGE-1 F1 score: 0.4516
Average ROUGE-L F1 score: 0.4113

Fine-Tuned Model Benchmark Results:
Corpus BLEU score: 0.4644
Average ROUGE-1 F1 score: 0.7211
Average ROUGE-L F1 score: 0.6707


Dựa trên kết quả benchmark, ta có thể so sánh như sau:

- **Mô hình Base (Qwen2.5-7B):**
  - Corpus BLEU score: 0.2564  
    (Tương đối thấp, chỉ khoảng 25,64% n-gram overlap với câu trả lời chuẩn.)
  - Average ROUGE-1 F1 score: 0.4516  
    (Chỉ khoảng 45,16% về độ bao phủ từ.)
  - Average ROUGE-L F1 score: 0.4113  
    (Khoảng 41,13% về độ phù hợp cấu trúc hoặc chuỗi từ chung.)

- **Mô hình Fine-Tuned:**
  - Corpus BLEU score: 0.4644  
    (Gấp gần 2 lần so với Base, cho thấy sự trùng khớp n-gram cao hơn.)
  - Average ROUGE-1 F1 score: 0.7211  
    (Độ bao phủ từ tăng lên tới khoảng 72,11%.)
  - Average ROUGE-L F1 score: 0.6707  
    (Độ tương đồng về chuỗi từ tăng lên đến 67,07%.)

**Nhận xét:**

- **So sánh trực tiếp:**  
  Mô hình fine-tuned cho kết quả cao hơn đáng kể trên tất cả các chỉ số so với mô hình base. Điều này chứng tỏ rằng quá trình fine-tuning đã giúp mô hình cải thiện khả năng tạo ra câu trả lời phù hợp, có nội dung trùng khớp và đầy đủ hơn với câu trả lời tham chiếu.
  
- **Ý nghĩa của các chỉ số:**  
  - **BLEU:** Cho biết mức độ trùng khớp n-gram giữa câu trả lời của mô hình và câu trả lời chuẩn. Sự tăng lên từ 0.2564 lên 0.4644 cho thấy fine-tuning giúp mô hình tái hiện lại cấu trúc và từ ngữ của câu trả lời chuẩn tốt hơn.
  - **ROUGE-1 & ROUGE-L:** Đánh giá về sự bao phủ nội dung và cấu trúc câu. Các chỉ số cao hơn của mô hình fine-tuned cho thấy mô hình này không chỉ bắt được từ khóa mà còn duy trì được thứ tự và cấu trúc của các câu trả lời chuẩn.

**Kết luận:**  
Quá trình fine-tuning đã cải thiện đáng kể hiệu suất của mô hình trên bộ dữ liệu ViMedAQA, từ đó tăng khả năng trả lời chính xác và đầy đủ các câu hỏi y tế so với mô hình gốc Qwen2.5-7B. Các kết quả này cho thấy việc chuyên biệt hóa mô hình cho lĩnh vực y tế có thể mang lại sự cải thiện rõ rệt về chất lượng câu trả lời.

In [ ]:
from unsloth import FastLanguageModel
# Specify the saved model directory
output_dir = "/home/ltnga/NguyenTrinhTest/fine_tuned_model_new"

# Load the fine-tuned model
model, tokenizer = FastLanguageModel.from_pretrained(output_dir)

# Ensure the prompt variable is defined
prompt = """
You are a helpful assistant. Your task is to merge two user inputs into one concise, specific question.
Do not output any internal reasoning; only output the final merged question.

### Instruction:
1. Read the original question and the user’s clarification.
2. Combine them into a single, clear, specific question.
3. Output only the final merged question without extra text.

### Original Question:
{}

### Clarification:
{}

### Final Merged Question:
"""
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
# Tokenize inputs for inference
inputs = tokenizer(
    [
        prompt.format(
            "Tiểu đường trong thai kỳ",  # Instruction
            "Nguyên nhân gây ra tiểu đường thai kỳ?",  # Input
            ""  # Response left blank for generation
        )
    ],
    return_tensors="pt"
).to("cuda")  # or "cpu" if GPU is unavailable

# Set up a streamer for generation
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

# Generate text
output = model.generate(**inputs, streamer=text_streamer, max_new_tokens=512)


==((====))==  Unsloth 2025.2.4: Fast Qwen2 patching. Transformers: 4.46.1.
   \\   /|    GPU: NVIDIA A100-SXM4-80GB. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.21s/it]


Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Biviantac có thể điều trị trướng bụng, đầy hơi không?

### Input:
Thuốc Biviantac được chỉ định để điều trị các trường hợp do tăng tiết acid quá mức như: - Khó tiêu, nóng rát hay đau vùng thượng vị. - Trướng bụng, đầy hơi, ợ nóng, ợ hơi hay ợ chua. - Tăng độ acid, đau rát dạ dày. - Các rối loạn thường gặp trong những bệnh lý loét dạ dày tá tràng, thực quản.

### Response:
Có, Biviantac có thể điều trị trướng bụng và đầy hơi.<|endoftext|>


In [7]:
# Save the fine-tuned model
output_dir = "fine_tuned_model_new"
model.save_pretrained(output_dir)

# Save the tokenizer
tokenizer.save_pretrained(output_dir)


('fine_tuned_model_new/tokenizer_config.json',
 'fine_tuned_model_new/special_tokens_map.json',
 'fine_tuned_model_new/vocab.json',
 'fine_tuned_model_new/merges.txt',
 'fine_tuned_model_new/added_tokens.json',
 'fine_tuned_model_new/tokenizer.json')

In [9]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer

# Specify the saved model directory
output_dir = "fine_tuned_model_new"

# Load the fine-tuned model
model, tokenizer = FastLanguageModel.from_pretrained(output_dir)


==((====))==  Unsloth 2024.12.2: Fast Qwen2 patching. Transformers:4.46.1.
   \\   /|    GPU: NVIDIA A100-SXM4-80GB. Max memory: 79.138 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [24]:
FastLanguageModel.for_inference(model)
prompt_template = """Below is a task to classify a question as either 'simple' or 'complex.'

Rules:
1. A "simple" question is:
   - Short and straightforward.
   - Focused on one topic or concept.
   - Does not include detailed explanations or multiple parts.

2. A "complex" question is:
   - Longer and includes multiple sentences or clauses.
   - Provides detailed context or background information.
   - Often includes multiple parts or follow-up queries.

Classify the following question according to these rules:

Question: {}
Answer:
{}
"""

# Example questions
questions = [
    "Biviantac có thể điều trị trướng bụng, đầy hơi không?",
    "Chào bác sĩ, Răng cháu hiện tại có mủ ở dưới lợi nhưng khi đau cháu sẽ không ngủ được (quá đau). Tuy nhiên chỉ vài ngày là hết mà thỉnh thoảng nó lại bị đau. Chị cháu bảo là trước chị cháu cũng bị như vậy chỉ là đau răng tuổi dậy thì thôi. Bác sĩ cho cháu hỏi đau răng kèm có mủ dưới lợi là bệnh gì? Cháu có cần đi chữa trị không? Cháu cảm ơn."
]

# Tokenizing the inputs
tokenized_inputs = tokenizer(
    [prompt_template.format(question, "") for question in questions],
    return_tensors="pt",
    padding=True,
    truncation=True
).to("cuda")  # Adjust to "cpu" if GPU is unavailable

# Generating the outputs
outputs = model.generate(**tokenized_inputs, max_new_tokens=200)

# Decoding and printing the results
for output in outputs:
    print(tokenizer.decode(output, skip_special_tokens=True).strip())


Below is a task to classify a question as either 'simple' or 'complex.'

Rules:
1. A "simple" question is:
   - Short and straightforward.
   - Focused on one topic or concept.
   - Does not include detailed explanations or multiple parts.

2. A "complex" question is:
   - Longer and includes multiple sentences or clauses.
   - Provides detailed context or background information.
   - Often includes multiple parts or follow-up queries.

Classify the following question according to these rules:

Question: Biviantac có thể điều trị trướng bụng, đầy hơi không?
Answer:

The question "Biviantac có thể điều trị trướng bụng, đầy hơi không?" is classified as 'complex' based on the rules provided. It is longer and includes multiple sentences or clauses, providing detailed context about the treatment of bloating and gas.
Below is a task to classify a question as either 'simple' or 'complex.'

Rules:
1. A "simple" question is:
   - Short and straightforward.
   - Focused on one topic or concept.


In [ ]:
# Ensure the prompt variable is defined
prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
# Tokenize inputs for inference
inputs = tokenizer(
    [
        prompt.format(
            "Hãy phân tách câu hỏi sau thành các câu hỏi nhỏ, rõ ràng và độc lập, không trùng lặp. Trả về tối đa 3 câu hỏi, mỗi câu hỏi trên một dòng. Không thêm văn bản khác.",  # Instruction
            "Các nguyên nhân, biểu hiện và cách chữa bệnh tiểu đường",  # Input
            ""  # Response left blank for generation
        )
    ],
    return_tensors="pt"
).to("cuda")  # or "cpu" if GPU is unavailable

# Set up a streamer for generation
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

# Generate text
output = model.generate(**inputs, streamer=text_streamer, max_new_tokens=512)


Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Hãy phân tách câu hỏi sau thành các câu hỏi nhỏ, rõ ràng và độc lập, không trùng lặp. Trả về tối đa 3 câu hỏi, mỗi câu hỏi trên một dòng. Không thêm văn bản khác.

### Input:
Các nguyên nhân, biểu hiện và cách chữa bệnh tiểu đường

### Response:
1. Nguyên nhân của bệnh tiểu đường là gì?
2. Biểu hiện của bệnh tiểu đường là gì?
3. Cách chữa bệnh tiểu đường như thế nào?<|endoftext|>


In [ ]:
import os
import pandas as pd
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

# Environment variables
os.environ['HUGGING_FACE_HUB_TOKEN'] = "hfkey"

# Load Stage 1 Fine-Tuned Model
max_seq_length = 2048  # Sequence length
dtype = None  # Auto-detect precision
load_in_4bit = True  # Use 4-bit quantization

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name  =  "/home/ltnga/NguyenTrinh/fine_tuned_model_new",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Apply LoRA configuration
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["instruct"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

from datasets import load_dataset
dataset = load_dataset("VTSNLP/instruct_general_dataset", split = "train", cache_dir="/home/ltnga/.cache/huggingface/datasets")
dataset = dataset.map(formatting_prompts_func, batched = True,)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=4,  # Increase parallelism for A100
    packing=False,  # Can make training 5x faster for short sequences.
    args=TrainingArguments(
        per_device_train_batch_size=16,  # Optimize for A100 memory
        gradient_accumulation_steps=2,  # Adjust accumulation to balance memory and speed
        warmup_steps=100,  # Increase for better stability on large models
        max_steps=1000,  # Extend training steps for more substantial runs
        learning_rate=1e-4,  # Adjust LR for A100 training dynamics
        fp16=False,
        bf16=True,
        logging_steps=10,  # Log less frequently to reduce I/O overhead
        optim="adamw_8bit",  # Efficient optimization for large-scale training
        weight_decay=0.01,
        lr_scheduler_type="cosine",  # Cosine scheduler for smoother convergence
        seed=3407,
        output_dir="outputs",
        report_to="none",  # Use this for WandB etc
        save_strategy="steps",  # Save checkpoints regularly
        save_steps=100,  # Adjust based on training steps
        save_total_limit=2,  # Limit checkpoints to save space
    ),
)
trainer_stats = trainer.train()
trainer.save_model("/home/ltnga/NguyenTrinh/model_stage2")
tokenizer.save_pretrained("/home/ltnga/NguyenTrinh/model_stage2")

==((====))==  Unsloth 2024.12.2: Fast Qwen2 patching. Transformers:4.46.1.
   \\   /|    GPU: NVIDIA A100-SXM4-80GB. Max memory: 79.138 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Already have LoRA adapters! We shall skip this step.
Map (num_proc=4): 100%|██████████| 4531804/4531804 [42:39<00:00, 1770.79 examples/s] 
max_steps is given, it will override any value given in num_train_epochs
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 4,531,804 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 16 | Gradient Accumulation steps = 2
\        /    Total batch size = 32 | Total steps = 1,000
 "-____-"     Number of trainable parameters = 161,480,704


Step,Training Loss
10,1.680300
20,1.608300
30,1.558800
40,1.512000
50,1.459600
60,1.485600
70,1.394100
80,1.414000
90,1.401500
100,1.394700


('/home/ltnga/NguyenTrinh/model_stage2/tokenizer_config.json',
 '/home/ltnga/NguyenTrinh/model_stage2/special_tokens_map.json',
 '/home/ltnga/NguyenTrinh/model_stage2/vocab.json',
 '/home/ltnga/NguyenTrinh/model_stage2/merges.txt',
 '/home/ltnga/NguyenTrinh/model_stage2/added_tokens.json',
 '/home/ltnga/NguyenTrinh/model_stage2/tokenizer.json')

In [ ]:
import os
import pandas as pd
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

# Environment variables
os.environ['HUGGING_FACE_HUB_TOKEN'] = "hfkey"

# Load Stage 1 Fine-Tuned Model
max_seq_length = 2048  # Sequence length
dtype = None  # Auto-detect precision
load_in_4bit = True  # Use 4-bit quantization

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/home/ltnga/NguyenTrinh/model_stage2",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Apply LoRA configuration
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["question"]
    inputs = examples["context"]
    outputs = examples["answer"]
    texts = []
    for instruction, inputs, outputs in zip(instructions, inputs, outputs):
        text = prompt.format(instruction, inputs, outputs) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}
pass

from datasets import load_dataset
dataset  = load_dataset("tmnam20/ViMedAQA", "all", split = "train",cache_dir="/home/ltnga/.cache/huggingface/datasets")
dataset = dataset.map(formatting_prompts_func, batched = True,)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=4,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=16,
        gradient_accumulation_steps=2,
        warmup_steps=100,
        max_steps=1000,
        learning_rate=1e-4,
        fp16=False,
        bf16=True,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
    ),
)

trainer_stats = trainer.train()
trainer.save_model("/home/ltnga/NguyenTrinh/model_stage3")
tokenizer.save_pretrained("/home/ltnga/NguyenTrinh/model_stage3")

==((====))==  Unsloth 2024.12.2: Fast Qwen2 patching. Transformers:4.46.1.
   \\   /|    GPU: NVIDIA A100-SXM4-80GB. Max memory: 79.138 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Already have LoRA adapters! We shall skip this step.
Map (num_proc=4): 100%|██████████| 39881/39881 [00:08<00:00, 4583.99 examples/s]
max_steps is given, it will override any value given in num_train_epochs
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 39,881 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 16 | Gradient Accumulation steps = 2
\        /    Total batch size = 32 | Total steps = 1,000
 "-____-"     Number of trainable parameters = 161,480,704


Step,Training Loss
10,1.502200
20,1.441600
30,1.458600
40,1.414500
50,1.400700
60,1.387600
70,1.409100
80,1.369700
90,1.370600
100,1.391200


('/home/ltnga/NguyenTrinh/model_stage3/tokenizer_config.json',
 '/home/ltnga/NguyenTrinh/model_stage3/special_tokens_map.json',
 '/home/ltnga/NguyenTrinh/model_stage3/vocab.json',
 '/home/ltnga/NguyenTrinh/model_stage3/merges.txt',
 '/home/ltnga/NguyenTrinh/model_stage3/added_tokens.json',
 '/home/ltnga/NguyenTrinh/model_stage3/tokenizer.json')

In [5]:
from unsloth import FastLanguageModel
from transformers import TextStreamer

# Specify the saved model directory
output_dir = "/home/tttung/NguyenTrinhTest/model_stage3"

# Load the fine-tuned model
model, tokenizer = FastLanguageModel.from_pretrained(output_dir)

# Enable faster inference
FastLanguageModel.for_inference(model)

# Generalized prompt template with chain-of-thought instructions.
# The model is instructed to internally consider all aspects of ambiguity without outputting its internal reasoning.
prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context.
Use an internal chain-of-thought process to analyze the query, but do not output any internal reasoning. Only provide the final answer.

### Instruction:
Determine if the following medical query is ambiguous. A query is considered ambiguous if it is overly broad, vague, or lacking important details necessary for a focused answer. In particular:
- Queries that mention a general condition without specifying subtypes, treatment methods, diagnostic details, or context should be classified as ambiguous.
- Queries that are very short or use generic terms without qualifiers are ambiguous.
- Conversely, queries that include both the medical condition and additional specific details (such as a particular treatment, subtype, symptom, or diagnostic approach) are considered specific enough.
Respond with "yes" if the query is ambiguous, or "no" if it is specific enough.

### Input:
{}

### Response:
"""
example_query = "Cách điều trị ung thư"  # "How to treat cancer" - should be ambiguous.
# Format the prompt using the example query
prompt = prompt_template.format(example_query)
# Tokenize inputs for inference
inputs = tokenizer([prompt], return_tensors="pt")
inputs = {k: v.to("cuda") for k, v in inputs.items()}

# Set up a text streamer for real-time output (optional)
text_streamer = TextStreamer(tokenizer)
# Generate the output using the model
output = model.generate(**inputs, max_new_tokens=1024)


==((====))==  Unsloth 2025.3.19: Fast Qwen2 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 2. Max memory: 79.138 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.3.19 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [7]:
from unsloth import FastLanguageModel
from transformers import TextStreamer

# Specify the saved model directory
output_dir = "/home/tttung/NguyenTrinhTest/model_stage3"

# Load the fine-tuned model
model, tokenizer = FastLanguageModel.from_pretrained(output_dir)

# Enable faster inference
FastLanguageModel.for_inference(model)

# Prompt template đã được điều chỉnh bằng tiếng Việt
prompt_template = (
    "Bạn là một chuyên gia y tế chuyên về điều trị bệnh.\n"
    "Nhiệm vụ của bạn là phân tích truy vấn của người dùng và đưa ra 5 câu hỏi làm rõ độc đáo nhằm thu thập các thông tin cần thiết để trả lời chính xác.\n"
    "Hãy tự động xác định những khía cạnh quan trọng cần được làm rõ dựa trên nội dung của truy vấn. Các khía cạnh có thể bao gồm (nhưng không giới hạn):\n"
    "- Loại bệnh hoặc tình trạng,\n"
    "- Các phương pháp điều trị cụ thể,\n"
    "- Triệu chứng chủ chốt,\n"
    "- Chi tiết về chẩn đoán và nơi điều trị,\n"
    "- Chi phí điều trị hoặc các yếu tố bối cảnh khác.\n"
    "Hãy đảm bảo rằng mỗi câu hỏi là độc đáo và không lặp lại, phản ánh các khía cạnh khác nhau của truy vấn.\n\n"
    "Truy vấn của người dùng:\n"
    "{query}\n\n"
    "Các câu hỏi làm rõ:"
)

def generate_clarifying_question(query: str) -> str:
    # Format the prompt with the ambiguous query
    prompt = prompt_template.format(query=query)
    
    # Tokenize inputs for inference
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    
    # Set up a text streamer for real-time output (optional)
    text_streamer = TextStreamer(tokenizer)
    
    # Generate the output using the model
    output = model.generate(**inputs, streamer=text_streamer, max_new_tokens=1024)
    
    # Decode the output and return the clarifying questions
    clarifying_questions = tokenizer.decode(output[0], skip_special_tokens=True).strip()
    return clarifying_questions

# Example ambiguous query
example_query = "Các biến chứng thường gặp trong thai kì "

# Generate clarifying questions for the example query
clarifying_questions = generate_clarifying_question(example_query)
print(clarifying_questions)


==((====))==  Unsloth 2025.3.19: Fast Qwen2 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 2. Max memory: 79.138 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Bạn là một chuyên gia y tế chuyên về điều trị bệnh.
Nhiệm vụ của bạn là phân tích truy vấn của người dùng và đưa ra 5 câu hỏi làm rõ độc đáo nhằm thu thập các thông tin cần thiết để trả lời chính xác.
Hãy tự động xác định những khía cạnh quan trọng cần được làm rõ dựa trên nội dung của truy vấn. Các khía cạnh có thể bao gồm (nhưng không giới hạn):
- Loại bệnh hoặc tình trạng,
- Các phương pháp điều trị cụ thể,
- Triệu chứng chủ chốt,
- Chi tiết về chẩn đoán và nơi điều trị,
- Chi phí điều trị hoặc các yếu tố bối cảnh khác.
Hã

In [6]:
from unsloth import FastLanguageModel
from transformers import TextStreamer
import re

# Specify the saved model directory
output_dir = "/home/tttung/NguyenTrinhTest/model_stage3"

# Load the fine-tuned model
model, tokenizer = FastLanguageModel.from_pretrained(output_dir)

# Enable faster inference
FastLanguageModel.for_inference(model)

# Prompt template for detecting ambiguity (in Vietnamese)
prompt_template = (
    "Below is an instruction that describes a task, paired with an input that provides further context.\n"
    "Use an internal chain-of-thought process to analyze the query, but do not output any internal reasoning. Only provide the final answer.\n\n"
    "### Instruction:\n"
    "Determine if the following query is ambiguous. A query should be classified as not ambiguous if it is clearly phrased as a question—for example, if it ends with 'là gì?', contains a question mark, or uses explicit interrogative phrases that request a definition or explanation—or if it includes multiple specific details and aspects (such as causes, symptoms, and treatment options) that provide sufficient context. Conversely, a query is considered ambiguous if it only mentions a single topic or condition without additional details, or if it is short and lacks context.\n"
    "Respond with 'yes' if the query is ambiguous, or 'no' if it is specific enough. Do not include any additional text.\n\n"
    "### Input:\n"
    "{query}\n\n"
    "### Response:\n"
)

def detect_ambiguity(query: str) -> bool:
    # Format the prompt with the query
    prompt = prompt_template.format(query=query)
    
    # Tokenize inputs for inference
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    
    # Set up a text streamer for real-time output (optional)
    text_streamer = TextStreamer(tokenizer)
    
    # Generate the output using the model
    output = model.generate(**inputs, streamer=text_streamer, max_new_tokens=64, do_sample=False)
    
    # Decode the output and check if it's ambiguous
    full_response = tokenizer.decode(output[0], skip_special_tokens=True).strip().lower()
    
    # Parse the response to determine if the query is ambiguous
    lines = [line.strip() for line in full_response.split("\n") if line.strip()]
    if not lines:
        return True  # Default to ambiguous if there's no clear response
    final_line = re.sub(r"[^\w\s]", "", lines[-1]).strip()
    if final_line == "yes":
        return True   # Ambiguous
    elif final_line == "no":
        return False  # Not ambiguous
    else:
        print(f"Could not parse a clear answer. Final line: '{final_line}'")
        return True   # Default to ambiguous

# Example ambiguous query
example_query = "Tiểu đường trong thai kì"

# Detect ambiguity for the example query
is_ambiguous = detect_ambiguity(example_query)
print(f"Is the query ambiguous? {'Yes' if is_ambiguous else 'No'}")


==((====))==  Unsloth 2025.3.19: Fast Qwen2 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 2. Max memory: 79.138 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Below is an instruction that describes a task, paired with an input that provides further context.
Use an internal chain-of-thought process to analyze the query, but do not output any internal reasoning. Only provide the final answer.

### Instruction:
Determine if the following query is ambiguous. A query should be classified as not ambiguous if it is clearly phrased as a question—for example, if it ends with 'là gì?', contains a question mark, or uses explicit interrogative phrases that request a definition or explanation—o

In [42]:
import pandas as pd
from transformers import TextStreamer
from unsloth import FastLanguageModel

# File paths
dataset_path = "/home/ltnga/NguyenTrinh/ViMedAQA_full_dataset.csv"
model_paths = ["Qwen/Qwen2.5-7B", "/home/ltnga/NguyenTrinh/outputs_stage3/final_model"]

# Prompt format
prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Define metrics
def calculate_exact_match(predictions, ground_truths):
    return sum(1 for p, g in zip(predictions, ground_truths) if p.strip() == g.strip()) / len(predictions)

def calculate_f1(predictions, ground_truths):
    def f1_score(pred, gt):
        pred_tokens = pred.split()
        gt_tokens = gt.split()
        common = set(pred_tokens) & set(gt_tokens)
        if not common:
            return 0
        precision = len(common) / len(pred_tokens)
        recall = len(common) / len(gt_tokens)
        return 2 * precision * recall / (precision + recall)

    return sum(f1_score(p, g) for p, g in zip(predictions, ground_truths)) / len(predictions)

# Load dataset
data = pd.read_csv(dataset_path)

# Limit to the first 2000 samples
data = data.head(1500)

# Evaluate each model
results = {}
for model_path in model_paths:
    print(f"Evaluating model: {model_path}")
    model, tokenizer = FastLanguageModel.from_pretrained(model_path)
    FastLanguageModel.for_inference(model)
    model.to("cuda")

    predictions = []
    text_streamer = TextStreamer(tokenizer)
    total_samples = len(data)

    for idx, row in data.iterrows():
        instruction = row["question"]
        input_text = row["context"]
        expected_response = row["answer"]
        formatted_prompt = prompt.format(instruction, input_text, "")
        inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
        output = model.generate(**inputs, streamer=text_streamer, max_new_tokens=256)
        generated_response = tokenizer.decode(output[0], skip_special_tokens=True)

        if "### Response:" in generated_response:
            generated_response = generated_response.split("### Response:")[1].strip()

        predictions.append(generated_response)
        progress = ((idx + 1) / total_samples) * 100
        print(f"Progress: {progress:.2f}% ({idx + 1}/{total_samples})", end="\r")

    print()

    em_score = calculate_exact_match(predictions, data["answer"])
    f1_score_value = calculate_f1(predictions, data["answer"])
    results[model_path] = {"Exact Match": em_score, "F1": f1_score_value}

for model_path, metrics in results.items():
    print(f"Results for model: {model_path}")
    print(f"Exact Match: {metrics['Exact Match']:.4f}")
    print(f"F1 Score: {metrics['F1']:.4f}")


KeyboardInterrupt: 

### Try to merge model

In [15]:
import os
from unsloth import FastLanguageModel
from peft import PeftModel
import torch

BASE_MODEL_PATH = "Qwen/Qwen2.5-7B-Instruct"
FINE_TUNED_MODEL_PATH = "outputs_stage3/final_model"
OUTPUT_PATH = "outputs_stage3/merged_model"

print("Loading base model...")
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_PATH,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

print("Loading fine-tuned LoRA adapter...")
fine_tuned_model = PeftModel.from_pretrained(base_model, FINE_TUNED_MODEL_PATH)

print("Merging LoRA weights into the base model...")
merged_model = fine_tuned_model.merge_and_unload()

print("Saving merged model and tokenizer...")
merged_model.save_pretrained(OUTPUT_PATH)
tokenizer.save_pretrained(OUTPUT_PATH)

print(f"Model merging complete! Merged model saved to: {OUTPUT_PATH}")


Loading base model...
==((====))==  Unsloth 2024.12.2: Fast Qwen2 patching. Transformers:4.46.3.
   \\   /|    GPU: NVIDIA A100-SXM4-80GB. Max memory: 79.138 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading fine-tuned LoRA adapter...
Merging LoRA weights into the base model...


/home/ltnga/.local/lib/python3.10/site-packages/peft/tuners/lora/bnb.py:355: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Saving merged model and tokenizer...
Model merging complete! Merged model saved to: outputs_stage3/merged_model


In [1]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

# Prepare a prompt
prompt = "Hello, how are you doing today?"
inputs = tokenizer(prompt, return_tensors="pt")

# Generate text
outputs = model.generate(**inputs, max_length=50, do_sample=True)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generated_text)


/home/tttung/NguyenTrinhTest/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 4 files: 100%|██████████| 4/4 [03:32<00:00, 53.08s/it] 
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Loading checkpoint shards: 100%|██████████| 4/4 [00:13<00:00,  3.46s/it]


Hello, how are you doing today? I'm feeling a bit down and could use some uplifting words of encouragement. Can you share any personal experiences or stories that have helped you overcome challenges and find hope in difficult times?
Of course, I'd be
